# 07 - YOLOv8s Training Preparation for Retail Theft Detection

**Purpose:** Configure everything for YOLOv8s training

**Objectives:**
- GPU verification and diagnostics
- Generate optimized data.yaml
- Configure training hyperparameters
- Set up class-weighted loss for theft detection
- Prepare training configuration file
- Provide expert recommendations

---

## 1. Setup and Imports

In [1]:
# Install required packages
!pip install ultralytics torch torchvision pyyaml pandas matplotlib --quiet

In [2]:
import os
import sys
import yaml
import json
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Basic imports successful!")

Basic imports successful!


In [3]:
# Import PyTorch and check GPU
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")
    print(f"GPU count: {torch.cuda.device_count()}")
    
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"  Memory: {props.total_memory / 1024**3:.1f} GB")
        print(f"  Compute Capability: {props.major}.{props.minor}")
else:
    print("\n[WARNING] No GPU detected! Training will be slow on CPU.")
    print("Please ensure CUDA is properly installed.")

PyTorch version: 2.9.1+cu130
CUDA available: True
CUDA version: 13.0
cuDNN version: 91200
GPU count: 1

GPU 0: NVIDIA GeForce GTX 1650
  Memory: 4.0 GB
  Compute Capability: 7.5


In [4]:
import sys
print(sys.executable)


c:\Users\shaho\AppData\Local\Programs\Python\Python310\python.exe


In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())


CUDA available: True
GPU name: NVIDIA GeForce GTX 1650
GPU count: 1


In [6]:
# Import Ultralytics
try:
    from ultralytics import YOLO
    import ultralytics
    print(f"Ultralytics version: {ultralytics.__version__}")
    print("Ultralytics imported successfully!")
except ImportError as e:
    print(f"Error importing ultralytics: {e}")
    print("Please run: pip install ultralytics")

Ultralytics version: 8.3.247
Ultralytics imported successfully!


In [7]:
# Configuration
BASE_DIR = Path(r"c:/Users/shaho/OneDrive - Nile University/Desktop/AletrixGrad")

# Determine which dataset to use (priority: processed > augmented > cleaned > original)
PROCESSED_DIR = BASE_DIR / "dataset_processed"
AUGMENTED_DIR = BASE_DIR / "dataset_augmented"
CLEANED_DIR = BASE_DIR / "dataset_cleaned"
ORIGINAL_DIR = BASE_DIR / "cc-tv-footage-annotation-b8-lcysc-b1-2"

for dataset_dir in [AUGMENTED_DIR, CLEANED_DIR, ORIGINAL_DIR]:
    if dataset_dir.exists() and (dataset_dir / "train" / "images").exists():
        DATASET_DIR = dataset_dir
        break
else:
    DATASET_DIR = ORIGINAL_DIR

print(f"Using dataset: {DATASET_DIR}")

CONFIGS_DIR = BASE_DIR / "configs"
OUTPUT_DIR = BASE_DIR / "outputs"
RUNS_DIR = BASE_DIR / "runs"

# Create directories
for d in [CONFIGS_DIR, OUTPUT_DIR, RUNS_DIR]:
    d.mkdir(exist_ok=True)

# Class definitions
CLASS_NAMES = {
    0: 'Customer-Bagpack',
    1: 'Product',
    2: 'Product-Picked',
    3: 'Shopping-Cart',
    4: 'normal',
    5: 'theft'
}
NUM_CLASSES = 6

Using dataset: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_augmented


## 2. GPU Diagnostics

In [8]:
# Comprehensive GPU diagnostics
print("="*70)
print("GPU DIAGNOSTICS")
print("="*70)

gpu_info = {
    'cuda_available': torch.cuda.is_available(),
    'cuda_version': None,
    'cudnn_version': None,
    'gpu_count': 0,
    'gpus': []
}

if torch.cuda.is_available():
    gpu_info['cuda_version'] = torch.version.cuda
    gpu_info['cudnn_version'] = str(torch.backends.cudnn.version())
    gpu_info['gpu_count'] = torch.cuda.device_count()
    
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        
        # Get current memory usage
        torch.cuda.set_device(i)
        memory_allocated = torch.cuda.memory_allocated(i) / 1024**3
        memory_reserved = torch.cuda.memory_reserved(i) / 1024**3
        
        gpu_data = {
            'id': i,
            'name': props.name,
            'total_memory_gb': round(props.total_memory / 1024**3, 2),
            'compute_capability': f"{props.major}.{props.minor}",
            'multi_processor_count': props.multi_processor_count,
            'memory_allocated_gb': round(memory_allocated, 2),
            'memory_reserved_gb': round(memory_reserved, 2)
        }
        gpu_info['gpus'].append(gpu_data)
        
        print(f"\nGPU {i}: {props.name}")
        print(f"  Total Memory:      {gpu_data['total_memory_gb']:.2f} GB")
        print(f"  Allocated Memory:  {gpu_data['memory_allocated_gb']:.2f} GB")
        print(f"  Reserved Memory:   {gpu_data['memory_reserved_gb']:.2f} GB")
        print(f"  Compute Capability: {gpu_data['compute_capability']}")
        print(f"  SMs (Multiprocessors): {gpu_data['multi_processor_count']}")
    
    # Recommend batch size based on GPU memory
    total_gpu_memory = sum(g['total_memory_gb'] for g in gpu_info['gpus'])
    
    if total_gpu_memory >= 24:
        recommended_batch = 32
    elif total_gpu_memory >= 16:
        recommended_batch = 16
    elif total_gpu_memory >= 8:
        recommended_batch = 8
    elif total_gpu_memory >= 6:
        recommended_batch = 4
    else:
        recommended_batch = 2
    
    print(f"\nRecommended batch size: {recommended_batch}")
    gpu_info['recommended_batch_size'] = recommended_batch
    
else:
    print("\n[CRITICAL WARNING]")
    print("No GPU detected! Training will be extremely slow.")
    print("\nPlease ensure:")
    print("  1. NVIDIA GPU is installed")
    print("  2. CUDA toolkit is installed")
    print("  3. PyTorch is installed with CUDA support")
    print("\nInstall PyTorch with CUDA:")
    print("  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118")
    gpu_info['recommended_batch_size'] = 4

# Save GPU info
with open(OUTPUT_DIR / 'gpu_diagnostics.json', 'w') as f:
    json.dump(gpu_info, f, indent=2)
print(f"\nGPU diagnostics saved to: {OUTPUT_DIR / 'gpu_diagnostics.json'}")

GPU DIAGNOSTICS

GPU 0: NVIDIA GeForce GTX 1650
  Total Memory:      4.00 GB
  Allocated Memory:  0.00 GB
  Reserved Memory:   0.00 GB
  Compute Capability: 7.5
  SMs (Multiprocessors): 14

Recommended batch size: 2

GPU diagnostics saved to: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\outputs\gpu_diagnostics.json


## 3. Analyze Dataset for Training Configuration

In [9]:
# Count class distribution for class weights
def get_class_distribution(dataset_path: Path) -> dict:
    """Count annotations per class."""
    class_counts = defaultdict(int)
    total_images = 0
    
    for split in ['train', 'valid', 'test']:
        labels_path = dataset_path / split / "labels"
        images_path = dataset_path / split / "images"
        
        if images_path.exists():
            total_images += len(list(images_path.glob("*.*")))
        
        if not labels_path.exists():
            continue
        
        for label_file in labels_path.glob("*.txt"):
            try:
                with open(label_file, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 1:
                            try:
                                class_id = int(parts[0])
                                class_counts[class_id] += 1
                            except ValueError:
                                pass
            except:
                pass
    
    return dict(class_counts), total_images

class_counts, total_images = get_class_distribution(DATASET_DIR)

print("="*70)
print("DATASET ANALYSIS FOR TRAINING")
print("="*70)
print(f"\nTotal images: {total_images:,}")
print(f"Total annotations: {sum(class_counts.values()):,}")

print("\nClass Distribution:")
for class_id in sorted(class_counts.keys()):
    count = class_counts[class_id]
    pct = count / sum(class_counts.values()) * 100
    print(f"  {CLASS_NAMES.get(class_id, f'Class {class_id}'):20s}: {count:6,} ({pct:5.1f}%)")

DATASET ANALYSIS FOR TRAINING

Total images: 2,803
Total annotations: 7,335

Class Distribution:
  Customer-Bagpack    :    605 (  8.2%)
  Product             :    793 ( 10.8%)
  Product-Picked      :    778 ( 10.6%)
  Shopping-Cart       :    137 (  1.9%)
  normal              :  4,626 ( 63.1%)
  theft               :    396 (  5.4%)


In [10]:
# Calculate class weights for imbalanced dataset
def calculate_class_weights(class_counts: dict, strategy: str = 'inverse_freq') -> dict:
    """Calculate class weights for handling imbalanced data.
    
    Strategies:
    - 'inverse_freq': Weight inversely proportional to frequency
    - 'effective_samples': Based on effective number of samples
    - 'sqrt_inverse': Square root of inverse frequency (smoother)
    """
    total = sum(class_counts.values())
    num_classes = len(class_counts)
    
    weights = {}
    
    if strategy == 'inverse_freq':
        for class_id, count in class_counts.items():
            weights[class_id] = total / (num_classes * count)
            
    elif strategy == 'sqrt_inverse':
        for class_id, count in class_counts.items():
            weights[class_id] = np.sqrt(total / (num_classes * count))
            
    elif strategy == 'effective_samples':
        beta = 0.9999  # Hyperparameter
        for class_id, count in class_counts.items():
            effective_num = (1 - beta**count) / (1 - beta)
            weights[class_id] = total / (num_classes * effective_num)
    
    # Normalize weights so mean is 1.0
    mean_weight = np.mean(list(weights.values()))
    weights = {k: v / mean_weight for k, v in weights.items()}
    
    return weights

# Calculate weights using sqrt_inverse (good balance)
class_weights = calculate_class_weights(class_counts, strategy='sqrt_inverse')

print("\n" + "="*70)
print("CLASS WEIGHTS FOR TRAINING")
print("="*70)
print("\nCalculated class weights (sqrt_inverse strategy):")
for class_id in sorted(class_weights.keys()):
    weight = class_weights[class_id]
    print(f"  {CLASS_NAMES.get(class_id, f'Class {class_id}'):20s}: {weight:.3f}")

# Special emphasis on theft class
theft_class_id = 5
print(f"\n[NOTE] Theft class weight: {class_weights.get(theft_class_id, 1.0):.3f}")
print("Higher weights mean more focus during training.")


CLASS WEIGHTS FOR TRAINING

Calculated class weights (sqrt_inverse strategy):
  Customer-Bagpack    : 0.930
  Product             : 0.812
  Product-Picked      : 0.820
  Shopping-Cart       : 1.953
  normal              : 0.336
  theft               : 1.149

[NOTE] Theft class weight: 1.149
Higher weights mean more focus during training.


## 4. Generate Optimized data.yaml

In [11]:
# Generate optimized data.yaml
data_yaml_content = {
    # Paths (absolute for reliability)
    'path': str(DATASET_DIR),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    
    # Classes
    'nc': NUM_CLASSES,
    'names': list(CLASS_NAMES.values())
}

# Save data.yaml
data_yaml_path = CONFIGS_DIR / 'data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False, sort_keys=False)

print("="*70)
print("DATA.YAML CONFIGURATION")
print("="*70)
print(f"\nSaved to: {data_yaml_path}")
print("\nContents:")
print("-"*40)
print(yaml.dump(data_yaml_content, default_flow_style=False, sort_keys=False))

DATA.YAML CONFIGURATION

Saved to: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\configs\data.yaml

Contents:
----------------------------------------
path: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\dataset_augmented
train: train/images
val: valid/images
test: test/images
nc: 6
names:
- Customer-Bagpack
- Product
- Product-Picked
- Shopping-Cart
- normal
- theft



## 5. Training Hyperparameter Configuration

In [12]:
# Determine optimal hyperparameters based on GPU and dataset
recommended_batch = gpu_info.get('recommended_batch_size', 8)

# Training configuration
training_config = {
    # Model
    'model': 'yolov8s.pt',  # YOLOv8 Small - good balance of speed/accuracy
    
    # Data
    'data': str(data_yaml_path),
    'imgsz': 640,  # Standard YOLOv8 input size
    
    # Training
    'epochs': 100,
    'batch': recommended_batch,
    'patience': 30,  # Early stopping patience
    
    # Optimizer
    'optimizer': 'AdamW',
    'lr0': 0.01,  # Initial learning rate
    'lrf': 0.01,  # Final learning rate factor
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    
    # Loss
    'box': 7.5,  # Box loss gain
    'cls': 0.5,  # Classification loss gain
    'dfl': 1.5,  # Distribution focal loss gain
    
    # Augmentation (retail surveillance optimized)
    'hsv_h': 0.015,  # HSV-Hue augmentation
    'hsv_s': 0.7,    # HSV-Saturation augmentation
    'hsv_v': 0.4,    # HSV-Value augmentation
    'degrees': 0.0,  # No rotation (surveillance cameras are fixed)
    'translate': 0.1,
    'scale': 0.5,
    'shear': 0.0,   # No shear (unrealistic for surveillance)
    'perspective': 0.0,  # No perspective (fixed camera)
    'flipud': 0.0,  # No vertical flip (unrealistic)
    'fliplr': 0.5,  # Horizontal flip OK
    'mosaic': 1.0,  # Enable mosaic augmentation
    'mixup': 0.0,   # Disable mixup (can confuse theft detection)
    'copy_paste': 0.0,
    
    # Hardware
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'workers': 8,
    'cache': True,  # Cache images for faster training
    'amp': True,    # Automatic mixed precision
    
    # Output
    'project': str(RUNS_DIR),
    'name': 'retail_theft_yolov8s',
    'exist_ok': True,
    'pretrained': True,
    'verbose': True,
    'seed': 42,
    
    # Validation
    'val': True,
    'plots': True,
    'save': True,
    'save_period': 10,  # Save checkpoint every 10 epochs
}

print("="*70)
print("TRAINING HYPERPARAMETERS")
print("="*70)

print("\n--- Model ---")
print(f"  Model: {training_config['model']}")
print(f"  Image Size: {training_config['imgsz']}")

print("\n--- Training ---")
print(f"  Epochs: {training_config['epochs']}")
print(f"  Batch Size: {training_config['batch']}")
print(f"  Early Stopping Patience: {training_config['patience']}")

print("\n--- Optimizer ---")
print(f"  Optimizer: {training_config['optimizer']}")
print(f"  Learning Rate: {training_config['lr0']}")
print(f"  Weight Decay: {training_config['weight_decay']}")
print(f"  Warmup Epochs: {training_config['warmup_epochs']}")

print("\n--- Hardware ---")
print(f"  Device: {training_config['device']}")
print(f"  Workers: {training_config['workers']}")
print(f"  Cache: {training_config['cache']}")
print(f"  AMP (Mixed Precision): {training_config['amp']}")

print("\n--- Augmentation ---")
print(f"  Mosaic: {training_config['mosaic']}")
print(f"  Horizontal Flip: {training_config['fliplr']}")
print(f"  Scale: {training_config['scale']}")

TRAINING HYPERPARAMETERS

--- Model ---
  Model: yolov8s.pt
  Image Size: 640

--- Training ---
  Epochs: 100
  Batch Size: 2
  Early Stopping Patience: 30

--- Optimizer ---
  Optimizer: AdamW
  Learning Rate: 0.01
  Weight Decay: 0.0005
  Warmup Epochs: 3

--- Hardware ---
  Device: 0
  Workers: 8
  Cache: True
  AMP (Mixed Precision): True

--- Augmentation ---
  Mosaic: 1.0
  Horizontal Flip: 0.5
  Scale: 0.5


In [13]:
# Save training configuration
config_path = CONFIGS_DIR / 'training_config.yaml'
with open(config_path, 'w') as f:
    yaml.dump(training_config, f, default_flow_style=False, sort_keys=False)

print(f"\nTraining configuration saved to: {config_path}")


Training configuration saved to: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\configs\training_config.yaml


## 6. Generate Training Script

In [14]:
# Generate training script
training_script = f'''#!/usr/bin/env python3
"""
YOLOv8s Training Script for Retail Theft Detection
Generated: {datetime.now().isoformat()}

Usage:
    python train_yolov8s.py
"""

import torch
from ultralytics import YOLO
from pathlib import Path

# Verify GPU
print("="*60)
print("GPU VERIFICATION")
print("="*60)
if torch.cuda.is_available():
    print(f"GPU: {{torch.cuda.get_device_name(0)}}")
    print(f"CUDA: {{torch.version.cuda}}")
    print(f"Memory: {{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}} GB")
else:
    print("[WARNING] No GPU detected! Training on CPU.")
print("="*60)

# Configuration
DATA_YAML = r"{str(data_yaml_path)}"
MODEL = "yolov8s.pt"
EPOCHS = {training_config['epochs']}
BATCH_SIZE = {training_config['batch']}
IMG_SIZE = {training_config['imgsz']}
DEVICE = 0 if torch.cuda.is_available() else "cpu"

def main():
    # Load model (pretrained on COCO)
    print("\\nLoading YOLOv8s model...")
    model = YOLO(MODEL)
    
    # Train
    print("\\nStarting training...")
    results = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        device=DEVICE,
        
        # Optimizer
        optimizer="AdamW",
        lr0=0.01,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        
        # Loss
        box=7.5,
        cls=0.5,
        dfl=1.5,
        
        # Augmentation (retail surveillance optimized)
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=0.0,
        translate=0.1,
        scale=0.5,
        shear=0.0,
        perspective=0.0,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.0,
        
        # Hardware
        workers=8,
        cache=True,
        amp=True,
        
        # Output
        project=r"{str(RUNS_DIR)}",
        name="retail_theft_yolov8s",
        exist_ok=True,
        pretrained=True,
        verbose=True,
        seed=42,
        
        # Validation
        val=True,
        plots=True,
        save=True,
        save_period=10,
        patience=30,
    )
    
    print("\\nTraining complete!")
    print(f"Results saved to: {{results.save_dir}}")
    
    return results

if __name__ == "__main__":
    main()
'''

# Save training script
script_path = BASE_DIR / 'train_yolov8s.py'
with open(script_path, 'w') as f:
    f.write(training_script)

print(f"Training script saved to: {script_path}")

Training script saved to: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\train_yolov8s.py


## 7. Performance Optimization Recommendations

In [15]:
# Expert recommendations
print("="*70)
print("PERFORMANCE OPTIMIZATION RECOMMENDATIONS")
print("="*70)

recommendations = f"""
1. MODEL SELECTION
   - Using YOLOv8s (small) for good balance of speed and accuracy
   - For higher accuracy: try YOLOv8m or YOLOv8l
   - For faster inference: use YOLOv8n

2. IMAGE SIZE CONSIDERATIONS
   - Current: 640x640 (standard)
   - For small objects (products): consider 1280x1280
   - Trade-off: larger size = better accuracy but slower training

3. AUGMENTATION FOR SURVEILLANCE
   - Enabled: Mosaic, horizontal flip, scale, HSV adjustments
   - Disabled: Rotation, shear, perspective (unrealistic for fixed cameras)
   - Consider: adding motion blur for realistic surveillance effects

4. CLASS IMBALANCE HANDLING
   - Theft class is underrepresented ({class_counts.get(5, 0):,} samples)
   - Strategies implemented:
     a) Data augmentation on minority classes
     b) Class weights calculated (theft weight: {class_weights.get(5, 1.0):.3f})
   - Additional options:
     a) Focal loss (increase gamma)
     b) More aggressive augmentation on theft samples

5. MAXIMIZING RECALL FOR THEFT CLASS
   - Use lower confidence threshold during inference (0.25 instead of 0.5)
   - Adjust cls_loss weight to emphasize classification
   - Consider post-processing with class-specific NMS thresholds
   - Fine-tune on theft-heavy subset after initial training

6. TRANSFER LEARNING STRATEGY
   - Start from yolov8s.pt (COCO pretrained)
   - First 10 epochs: freeze backbone, train only head
   - Remaining epochs: unfreeze all layers
   - Alternative: use smaller lr for pretrained layers

7. HARDWARE OPTIMIZATION
   - Batch size: {recommended_batch} (based on {gpu_info.get('gpus', [{}])[0].get('total_memory_gb', 'N/A')} GB GPU)
   - Enable AMP (automatic mixed precision) for faster training
   - Use cache=True for faster data loading
   - Workers: 8 (adjust based on CPU cores)

8. EARLY STOPPING
   - Patience: 30 epochs (stop if no improvement)
   - Monitor: mAP@0.5 and recall for theft class
   - Save best model based on validation performance

9. POST-TRAINING OPTIMIZATION
   - Export to ONNX/TensorRT for faster inference
   - Quantization (INT8) for edge deployment
   - Test with various confidence thresholds

10. RECOMMENDED TRAINING SCHEDULE
    Phase 1: Initial training (50-100 epochs)
    Phase 2: Fine-tune on hard examples (20-30 epochs)
    Phase 3: Optimize for recall if needed (10-20 epochs)
"""

print(recommendations)

# Save recommendations
with open(OUTPUT_DIR / 'training_recommendations.txt', 'w') as f:
    f.write(recommendations)
print(f"\nRecommendations saved to: {OUTPUT_DIR / 'training_recommendations.txt'}")

PERFORMANCE OPTIMIZATION RECOMMENDATIONS

1. MODEL SELECTION
   - Using YOLOv8s (small) for good balance of speed and accuracy
   - For higher accuracy: try YOLOv8m or YOLOv8l
   - For faster inference: use YOLOv8n

2. IMAGE SIZE CONSIDERATIONS
   - Current: 640x640 (standard)
   - For small objects (products): consider 1280x1280
   - Trade-off: larger size = better accuracy but slower training

3. AUGMENTATION FOR SURVEILLANCE
   - Enabled: Mosaic, horizontal flip, scale, HSV adjustments
   - Disabled: Rotation, shear, perspective (unrealistic for fixed cameras)
   - Consider: adding motion blur for realistic surveillance effects

4. CLASS IMBALANCE HANDLING
   - Theft class is underrepresented (396 samples)
   - Strategies implemented:
     a) Data augmentation on minority classes
     b) Class weights calculated (theft weight: 1.149)
   - Additional options:
     a) Focal loss (increase gamma)
     b) More aggressive augmentation on theft samples

5. MAXIMIZING RECALL FOR THEFT CLAS

## 8. Quick Start Commands

In [16]:
# Generate quick start commands
print("="*70)
print("QUICK START COMMANDS")
print("="*70)

commands = f"""
# Option 1: Run training script
python "{script_path}"

# Option 2: Train from command line (Ultralytics CLI)
yolo detect train \\
    data="{data_yaml_path}" \\
    model=yolov8s.pt \\
    epochs=100 \\
    batch={recommended_batch} \\
    imgsz=640 \\
    device=0 \\
    project="{RUNS_DIR}" \\
    name=retail_theft_yolov8s

# Option 3: Train in Python
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data='{data_yaml_path}', epochs=100, batch={recommended_batch})

# Validate trained model
yolo detect val model=runs/retail_theft_yolov8s/weights/best.pt data="{data_yaml_path}"

# Run inference on test images
yolo detect predict model=runs/retail_theft_yolov8s/weights/best.pt source="{DATASET_DIR}/test/images"
"""

print(commands)

# Save commands
with open(OUTPUT_DIR / 'training_commands.txt', 'w') as f:
    f.write(commands)
print(f"\nCommands saved to: {OUTPUT_DIR / 'training_commands.txt'}")

QUICK START COMMANDS

# Option 1: Run training script
python "c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\train_yolov8s.py"

# Option 2: Train from command line (Ultralytics CLI)
yolo detect train \
    data="c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\configs\data.yaml" \
    model=yolov8s.pt \
    epochs=100 \
    batch=2 \
    imgsz=640 \
    device=0 \
    project="c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\runs" \
    name=retail_theft_yolov8s

# Option 3: Train in Python
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data='c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\configs\data.yaml', epochs=100, batch=2)

# Validate trained model
yolo detect val model=runs/retail_theft_yolov8s/weights/best.pt data="c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\configs\data.yaml"

# Run inference on test images
yolo detect predict model=runs/retail_theft_yolov8s/weights/best.pt source="c

## 9. Final Summary

In [17]:
# Save comprehensive preparation report
preparation_report = {
    'timestamp': datetime.now().isoformat(),
    'dataset': {
        'path': str(DATASET_DIR),
        'total_images': total_images,
        'class_distribution': {CLASS_NAMES[k]: v for k, v in class_counts.items()}
    },
    'gpu_info': gpu_info,
    'training_config': training_config,
    'class_weights': {CLASS_NAMES[k]: round(v, 3) for k, v in class_weights.items()},
    'files_generated': [
        str(data_yaml_path),
        str(config_path),
        str(script_path)
    ]
}

report_path = OUTPUT_DIR / 'training_preparation_report.json'
with open(report_path, 'w') as f:
    json.dump(preparation_report, f, indent=2)

print("="*70)
print("TRAINING PREPARATION COMPLETE")
print("="*70)

print(f"""
SUMMARY:
--------
Dataset: {DATASET_DIR.name}
Total Images: {total_images:,}
Model: YOLOv8s (Small)
GPU: {gpu_info['gpus'][0]['name'] if gpu_info['gpus'] else 'CPU'}
Batch Size: {training_config['batch']}

FILES GENERATED:
  - data.yaml: {data_yaml_path}
  - Training config: {config_path}
  - Training script: {script_path}
  - Preparation report: {report_path}

NEXT STEPS:
  1. Review configuration files
  2. Run: python "{script_path}"
  3. Or proceed to 08_training_evaluation.ipynb
""")

print("="*70)

TRAINING PREPARATION COMPLETE

SUMMARY:
--------
Dataset: dataset_augmented
Total Images: 2,803
Model: YOLOv8s (Small)
GPU: NVIDIA GeForce GTX 1650
Batch Size: 2

FILES GENERATED:
  - data.yaml: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\configs\data.yaml
  - Training config: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\configs\training_config.yaml
  - Training script: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\train_yolov8s.py
  - Preparation report: c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\outputs\training_preparation_report.json

NEXT STEPS:
  1. Review configuration files
  2. Run: python "c:\Users\shaho\OneDrive - Nile University\Desktop\AletrixGrad\train_yolov8s.py"
  3. Or proceed to 08_training_evaluation.ipynb



---

## Next Steps

1. Review the generated configuration files
2. Run the training script: `python train_yolov8s.py`
3. Or proceed to **08_training_evaluation.ipynb** to train interactively

---